# 🔍 Module 5.3: Feature Extraction with CNNs

## What Do CNNs Actually Learn? — From Layer Representations to Grad-CAM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_05_Deep_Learning_For_Images/03_Feature_Extraction/feature_extraction.ipynb)

---

## 🎯 Learning Objectives

1. Use CNNs as feature extractors by tapping intermediate layers
2. Visualize and interpret what each CNN layer learns
3. Implement Grad-CAM to see where the network "looks"
4. Understand feature vectors as state representations for RL

---

In [ ]:
# Google Colab Setup
!pip install torch torchvision numpy matplotlib scikit-learn -q

In [ ]:
# Download REAL open-source datasets (TINY - under 250MB total)
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# MNIST (handwritten digits - 11MB)
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# CIFAR-10 (real photographs - 170MB)
transform_cifar = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar)

# FashionMNIST (clothing items - 30MB, great for transfer learning!)
fashion_train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
fashion_test = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

print(f"✅ MNIST: {len(mnist_train)} train + {len(mnist_test)} test (28x28 grayscale)")
print(f"✅ CIFAR-10: {len(cifar_train)} train + {len(cifar_test)} test (32x32 RGB)")
print(f"✅ FashionMNIST: {len(fashion_train)} train + {len(fashion_test)} test (28x28)")
print(f"   Classes: {cifar_train.classes}")
print(f"\n📦 Total download: ~211MB (NO STL-10 needed!)")

## Deep Derivation: Feature Extraction via Learned Representations

### Step 1: Feature Maps as Learned Basis Functions
A convolutional layer computes feature maps $\mathbf{h}^{(l)}$:
$$h^{(l)}_k(i,j) = \sigma\!\left(\sum_{c=1}^{C_{l-1}} \sum_{m,n} W^{(l)}_{k,c}(m,n) \cdot h^{(l-1)}_c(i+m, j+n) + b^{(l)}_k\right)$$

Each filter $W^{(l)}_k$ acts as a **learned template detector**; the feature map $h^{(l)}_k$ is the spatial response to that template.

### Step 2: Why Deep Features are Hierarchical (Representation Theorem)
**Claim:** Composing layers creates increasingly abstract features.

Layer 1 detects edges: $h^{(1)} = \sigma(W^{(1)} * X)$

Layer 2 combines edges into textures: $h^{(2)} = \sigma(W^{(2)} * h^{(1)})$

**Proof (by receptive field growth):**
$$r_l = 1 + \sum_{i=1}^{l} (k_i - 1)\prod_{j=1}^{i-1} s_j$$

Each deeper layer integrates information over a larger spatial region, enabling detection of increasingly complex patterns. $\blacksquare$

### Step 3: Global Average Pooling (GAP) — Dimensionality Reduction
Given feature maps $h_k \in \mathbb{R}^{H \times W}$, GAP computes:
$$\bar{h}_k = \frac{1}{HW}\sum_{i=1}^{H}\sum_{j=1}^{W} h_k(i,j)$$

**Proof GAP is shift-invariant:**

Let $T_\tau h_k(i,j) = h_k(i-\tau_1, j-\tau_2)$ with circular boundary. Then:
$$\frac{1}{HW}\sum_{i,j} T_\tau h_k(i,j) = \frac{1}{HW}\sum_{i,j} h_k(i-\tau_1, j-\tau_2) = \frac{1}{HW}\sum_{i',j'} h_k(i',j') = \bar{h}_k \quad \blacksquare$$

GAP reduces $C \times H \times W$ features to a $C$-dimensional vector, providing translation invariance.

### Step 4: Feature Similarity via Cosine Distance
$$\text{sim}(\mathbf{f}_1, \mathbf{f}_2) = \frac{\mathbf{f}_1 \cdot \mathbf{f}_2}{\|\mathbf{f}_1\| \|\mathbf{f}_2\|} = \cos\theta$$

**Proof cosine similarity is bounded $[-1, 1]$:**

By Cauchy-Schwarz: $|\mathbf{f}_1 \cdot \mathbf{f}_2| \leq \|\mathbf{f}_1\|\|\mathbf{f}_2\|$

Dividing both sides: $\left|\frac{\mathbf{f}_1 \cdot \mathbf{f}_2}{\|\mathbf{f}_1\|\|\mathbf{f}_2\|}\right| \leq 1 \quad \blacksquare$

### Step 5: PCA on Feature Vectors (Optimal Linear Projection)
Given features $\mathbf{F} \in \mathbb{R}^{N \times D}$ (centered), find $\mathbf{W} \in \mathbb{R}^{D \times d}$ minimizing reconstruction error:

$$\min_{\mathbf{W}} \|\mathbf{F} - \mathbf{F}\mathbf{W}\mathbf{W}^T\|_F^2 \quad \text{s.t. } \mathbf{W}^T\mathbf{W} = \mathbf{I}_d$$

**Solution:** $\mathbf{W} = [\mathbf{v}_1, \ldots, \mathbf{v}_d]$, top-$d$ eigenvectors of $\mathbf{F}^T\mathbf{F}$.

**Proof:** By the Eckart-Young theorem, the best rank-$d$ approximation of $\mathbf{F}$ in Frobenius norm is given by truncated SVD, whose right singular vectors are the top eigenvectors of $\mathbf{F}^T\mathbf{F}$. $\blacksquare$

### Step 6: Variance Explained
$$\text{Variance ratio}_k = \frac{\lambda_k}{\sum_{i=1}^D \lambda_i}$$

Choosing $d$ such that $\sum_{k=1}^d \lambda_k / \sum_{i=1}^D \lambda_i \geq 0.95$ preserves 95% of information.

### Step 7: Connection to RL — Feature-Based State Representation
In RL for vision, the state $s_t$ is a CNN feature vector rather than raw pixels:
$$s_t = f_\theta(\text{Image}_t) \in \mathbb{R}^d, \quad d \ll H \times W \times C$$

This dramatically reduces the state space: from $256^{H \times W \times C}$ (raw pixels) to a continuous $d$-dimensional manifold, making Q-learning and policy gradient methods tractable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms, datasets
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('All libraries loaded!')

---

## Part 1: CNN as a Feature Extractor

### 1.1 The Key Idea

A CNN trained for classification can be decomposed into two parts:

$$\underbrace{\text{Conv layers} \rightarrow \text{Pool layers}}_{\text{Feature extractor } \phi(\mathbf{x})} \rightarrow \underbrace{\text{FC layers} \rightarrow \text{Softmax}}_{\text{Classifier } f(\phi(\mathbf{x}))}$$

The feature extractor $\phi: \mathbb{R}^{C \times H \times W} \rightarrow \mathbb{R}^d$ maps high-dimensional images into compact feature vectors. These vectors capture **semantic information** about the image content.

### 1.2 Why This Matters for RL

In Deep RL, the agent's **state** is often a raw image (e.g., Atari game frame). Using $\phi(\mathbf{x})$ as the state representation is far more efficient than using raw pixels:

$$Q(s, a) \approx Q(\phi(\text{image}), a; \theta)$$

The CNN extracts meaningful features; the RL agent learns from those features.

In [ ]:
# Load a pre-trained ResNet18 (we'll use this throughout)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.eval()
resnet = resnet.to(device)

# ImageNet preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('ResNet18 architecture (feature extraction parts):')
print('='*60)
for name, module in list(resnet.named_children())[:8]:
    if hasattr(module, 'weight'):
        print(f'  {name:12s}: {module.__class__.__name__:20s} params={sum(p.numel() for p in module.parameters()):,}')
    else:
        total_p = sum(p.numel() for p in module.parameters()) if list(module.parameters()) else 0
        print(f'  {name:12s}: {module.__class__.__name__:20s} params={total_p:,}')

total = sum(p.numel() for p in resnet.parameters())
print(f'\nTotal parameters: {total:,}')

### 1.3 Extracting Features from Intermediate Layers

We use PyTorch **hooks** to capture activations at any layer during the forward pass:

In [ ]:
class FeatureExtractor:
    """Extract features from any intermediate layer of a CNN."""

    def __init__(self, model, target_layers):
        self.model = model
        self.features = {}
        self.hooks = []

        for name, module in model.named_modules():
            if name in target_layers:
                hook = module.register_forward_hook(self._make_hook(name))
                self.hooks.append(hook)

    def _make_hook(self, name):
        def hook_fn(module, input, output):
            self.features[name] = output.detach()
        return hook_fn

    def __call__(self, x):
        self.features = {}
        output = self.model(x)
        return output, self.features

    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()


# Create synthetic test images using colored patterns
def create_test_images():
    """Create simple synthetic test images."""
    images = []
    titles = []

    # Horizontal stripes
    img = np.zeros((224, 224, 3), dtype=np.float32)
    for i in range(0, 224, 20):
        img[i:i+10, :] = [1.0, 0.0, 0.0]
    images.append(img)
    titles.append('Horizontal Stripes')

    # Vertical stripes
    img = np.zeros((224, 224, 3), dtype=np.float32)
    for j in range(0, 224, 20):
        img[:, j:j+10] = [0.0, 0.0, 1.0]
    images.append(img)
    titles.append('Vertical Stripes')

    # Diagonal gradient
    x_grad = np.linspace(0, 1, 224)
    y_grad = np.linspace(0, 1, 224)
    xx, yy = np.meshgrid(x_grad, y_grad)
    img = np.stack([xx, yy, 1 - xx], axis=-1).astype(np.float32)
    images.append(img)
    titles.append('Color Gradient')

    # Circle
    img = np.zeros((224, 224, 3), dtype=np.float32)
    cx, cy = 112, 112
    for i in range(224):
        for j in range(224):
            if (i - cx)**2 + (j - cy)**2 < 60**2:
                img[i, j] = [0.0, 1.0, 0.0]
    images.append(img)
    titles.append('Green Circle')

    return images, titles

test_images, test_titles = create_test_images()
print(f'Created {len(test_images)} test images')

In [ ]:
# Extract features at different layers
target_layers = ['conv1', 'layer1', 'layer2', 'layer3', 'layer4']
extractor = FeatureExtractor(resnet, target_layers)

fig, axes = plt.subplots(len(test_images), len(target_layers) + 1, figsize=(22, 4 * len(test_images)))

for img_idx, (img, title) in enumerate(zip(test_images, test_titles)):
    axes[img_idx, 0].imshow(img)
    axes[img_idx, 0].set_title(title, fontsize=11)
    axes[img_idx, 0].axis('off')

    # Preprocess and run
    x = torch.tensor(img.transpose(2, 0, 1)).unsqueeze(0).to(device)
    # Normalize
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
    x_norm = (x - mean) / std

    _, features = extractor(x_norm)

    for layer_idx, layer_name in enumerate(target_layers):
        feat = features[layer_name].cpu().numpy()[0]
        # Show mean activation across channels
        mean_activation = feat.mean(axis=0)
        axes[img_idx, layer_idx + 1].imshow(mean_activation, cmap='hot')
        axes[img_idx, layer_idx + 1].set_title(
            f'{layer_name}\n{feat.shape[0]}ch × {feat.shape[1]}×{feat.shape[2]}',
            fontsize=9
        )
        axes[img_idx, layer_idx + 1].axis('off')

plt.suptitle('Feature Representations at Different CNN Depths\n'
             '(mean activation across channels at each layer)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Observations:')
print('  - Early layers (conv1, layer1): Preserve spatial details, detect edges/textures')
print('  - Middle layers (layer2, layer3): Capture patterns and parts')
print('  - Deep layers (layer4): Abstract, semantic representations')

---

## Part 2: What Each Layer Learns

### 2.1 Mathematical Perspective

Each convolutional layer learns a set of filters $\{\mathbf{W}_k\}_{k=1}^{C_{\text{out}}}$. The activation of filter $k$ at position $(i,j)$ is:

$$a_k(i,j) = g\left(\sum_{c=1}^{C_{\text{in}}} \sum_{m,n} W_k(c,m,n) \cdot x(c, i+m, j+n) + b_k\right)$$

This is a **template matching** operation: the filter $\mathbf{W}_k$ responds maximally to input patterns that match its learned weights.

### 2.2 Layer-by-Layer Analysis

| Layer | What It Learns | Receptive Field | Feature Dimension |
|-------|---------------|-----------------|------------------|
| Conv1 | Edges, colors, gradients | 7×7 | 64 × 112 × 112 |
| Layer1 | Corners, textures | ~35×35 | 64 × 56 × 56 |
| Layer2 | Object parts, patterns | ~91×91 | 128 × 28 × 28 |
| Layer3 | Object parts, arrangements | ~187×187 | 256 × 14 × 14 |
| Layer4 | Objects, scenes | 224×224+ | 512 × 7 × 7 |

In [ ]:
# Visualize first-layer filters (conv1 of ResNet18)
conv1_weights = resnet.conv1.weight.detach().cpu().numpy()  # (64, 3, 7, 7)

fig, axes = plt.subplots(4, 16, figsize=(20, 6))
for i, ax in enumerate(axes.flat):
    if i < conv1_weights.shape[0]:
        w = conv1_weights[i].transpose(1, 2, 0)  # (7, 7, 3)
        # Normalize to [0, 1] for display
        w = (w - w.min()) / (w.max() - w.min() + 1e-8)
        ax.imshow(w)
    ax.axis('off')

plt.suptitle('ResNet18 Conv1 Learned Filters (7×7×3)\n'
             'These detect edges, gradients, and color patterns',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Conv1 has {conv1_weights.shape[0]} filters of size {conv1_weights.shape[1:]}') 
print('Many resemble Gabor filters and edge detectors — learned, not hand-designed!')

### 2.3 Channel Activation Analysis

Which filters activate most strongly for different image patterns?

In [ ]:
def analyze_channel_activations(model, extractor, images, titles, layer_name='layer2'):
    """Compare which channels activate for different images."""
    all_activations = []

    for img in images:
        x = torch.tensor(img.transpose(2, 0, 1)).unsqueeze(0).to(device)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
        x_norm = (x - mean) / std

        _, features = extractor(x_norm)
        feat = features[layer_name].cpu().numpy()[0]  # (C, H, W)
        channel_means = feat.mean(axis=(1, 2))  # mean activation per channel
        all_activations.append(channel_means)

    all_activations = np.array(all_activations)  # (n_images, n_channels)

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))

    # Channel activation comparison
    for i, title in enumerate(titles):
        axes[0].plot(all_activations[i], label=title, alpha=0.8, linewidth=1.5)
    axes[0].set_xlabel('Channel Index')
    axes[0].set_ylabel('Mean Activation')
    axes[0].set_title(f'Channel Activations at {layer_name}')
    axes[0].legend(fontsize=9)

    # Correlation matrix between image activations
    corr = np.corrcoef(all_activations)
    im = axes[1].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    axes[1].set_xticks(range(len(titles)))
    axes[1].set_yticks(range(len(titles)))
    axes[1].set_xticklabels(titles, rotation=30, ha='right', fontsize=9)
    axes[1].set_yticklabels(titles, fontsize=9)
    axes[1].set_title(f'Feature Similarity at {layer_name}')
    for i in range(len(titles)):
        for j in range(len(titles)):
            axes[1].text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center', fontsize=9)
    plt.colorbar(im, ax=axes[1])

    plt.suptitle(f'What Different Images Look Like to {layer_name}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

analyze_channel_activations(resnet, extractor, test_images, test_titles, 'layer2')
analyze_channel_activations(resnet, extractor, test_images, test_titles, 'layer4')

---

## Part 3: Grad-CAM — Where Does the Network Look?

### 3.1 Mathematical Foundation

**Grad-CAM** (Gradient-weighted Class Activation Mapping) highlights which spatial regions of the input are important for a particular class prediction.

Given a target class $c$, the Grad-CAM heatmap is computed as:

**Step 1:** Compute importance weights for each feature map channel $k$:

$$\alpha_k^c = \underbrace{\frac{1}{Z} \sum_{i} \sum_{j}}_{\text{global avg pool}} \underbrace{\frac{\partial y^c}{\partial A_{ij}^k}}_{\text{gradient of class score w.r.t. feature map}}$$

where $A^k$ is the $k$-th feature map and $y^c$ is the score for class $c$ (before softmax).

**Step 2:** Compute the weighted combination followed by ReLU:

$$L_{\text{Grad-CAM}}^c = \text{ReLU}\left(\sum_{k} \alpha_k^c A^k\right)$$

The ReLU ensures we only highlight positive contributions (features that increase the class score).

In [ ]:
class GradCAM:
    """
    Gradient-weighted Class Activation Mapping.
    Highlights important spatial regions for a given class prediction.
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        for name, module in model.named_modules():
            if name == target_layer:
                module.register_forward_hook(self._save_activation)
                module.register_full_backward_hook(self._save_gradient)
                break

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        """
        Generate Grad-CAM heatmap.

        Returns:
            heatmap: numpy array of shape (H, W) with values in [0, 1]
            predicted_class: the class the network predicted
        """
        self.model.eval()
        self.model.zero_grad()

        output = self.model(input_tensor)
        predicted_class = output.argmax(dim=1).item()

        if target_class is None:
            target_class = predicted_class

        # Backprop for target class
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1
        output.backward(gradient=one_hot, retain_graph=True)

        # Compute channel importance weights: alpha_k = GAP(gradients)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)

        # Weighted combination of feature maps
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, H, W)

        # ReLU: only positive contributions
        cam = F.relu(cam)

        # Normalize to [0, 1]
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = cam / cam.max()

        return cam, predicted_class

print('GradCAM class defined!')

In [ ]:
# Apply Grad-CAM to our test images
grad_cam = GradCAM(resnet, 'layer4')

fig, axes = plt.subplots(len(test_images), 3, figsize=(14, 4 * len(test_images)))

for idx, (img, title) in enumerate(zip(test_images, test_titles)):
    x = torch.tensor(img.transpose(2, 0, 1)).unsqueeze(0).to(device)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
    x_norm = (x - mean) / std
    x_norm.requires_grad_(True)

    cam, pred_class = grad_cam.generate(x_norm)

    # Resize CAM to input size
    cam_resized = np.array(
        torch.nn.functional.interpolate(
            torch.tensor(cam).unsqueeze(0).unsqueeze(0).float(),
            size=(224, 224), mode='bilinear', align_corners=False
        ).squeeze().numpy()
    )

    # Original
    axes[idx, 0].imshow(img)
    axes[idx, 0].set_title(f'{title}', fontsize=11)
    axes[idx, 0].axis('off')

    # Grad-CAM heatmap
    axes[idx, 1].imshow(cam_resized, cmap='jet')
    axes[idx, 1].set_title(f'Grad-CAM (class {pred_class})', fontsize=11)
    axes[idx, 1].axis('off')

    # Overlay
    axes[idx, 2].imshow(img)
    axes[idx, 2].imshow(cam_resized, cmap='jet', alpha=0.5)
    axes[idx, 2].set_title('Overlay', fontsize=11)
    axes[idx, 2].axis('off')

plt.suptitle('Grad-CAM: Where Does the Network Focus?\n'
             '(Red = high importance, Blue = low importance)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.2 Grad-CAM at Different Layers

Different layers focus on features at different scales:

In [ ]:
layers_to_visualize = ['layer1', 'layer2', 'layer3', 'layer4']
img = test_images[3]  # Circle image

fig, axes = plt.subplots(1, len(layers_to_visualize) + 1, figsize=(20, 4))

axes[0].imshow(img)
axes[0].set_title('Original', fontsize=12)
axes[0].axis('off')

for i, layer_name in enumerate(layers_to_visualize):
    gc = GradCAM(resnet, layer_name)

    x = torch.tensor(img.transpose(2, 0, 1)).unsqueeze(0).to(device)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
    x_norm = (x - mean) / std
    x_norm.requires_grad_(True)

    cam, _ = gc.generate(x_norm)
    cam_resized = torch.nn.functional.interpolate(
        torch.tensor(cam).unsqueeze(0).unsqueeze(0).float(),
        size=(224, 224), mode='bilinear', align_corners=False
    ).squeeze().numpy()

    axes[i + 1].imshow(img)
    axes[i + 1].imshow(cam_resized, cmap='jet', alpha=0.5)
    axes[i + 1].set_title(f'{layer_name}', fontsize=12)
    axes[i + 1].axis('off')

plt.suptitle('Grad-CAM at Different Depths: From Local to Global Attention',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Early layers focus on local features (edges of the circle).')
print('Deeper layers attend to the overall object.')

---

## Part 4: Feature Vectors for RL State Representation

### 4.1 From Images to Feature Vectors

For RL, we need a compact vector $\phi(\mathbf{x}) \in \mathbb{R}^d$ representing the state. A common approach:

$$\phi(\mathbf{x}) = \text{GlobalAvgPool}\left(\text{CNN}_{\text{features}}(\mathbf{x})\right) \in \mathbb{R}^{512}$$

This maps any image to a fixed-size 512-dimensional vector.

### 4.2 Properties of Good State Features

For RL, features should be:
1. **Compact**: Low-dimensional to speed up RL learning
2. **Discriminative**: Different states should have different features
3. **Smooth**: Similar states should have similar features (for generalization)
4. **Task-relevant**: Capture information needed for decision-making

In [ ]:
def extract_feature_vector(model, image_tensor):
    """Extract the 512-d feature vector from ResNet18 (before the FC layer)."""
    model.eval()
    with torch.no_grad():
        x = model.conv1(image_tensor)
        x = model.bn1(x)
        x = model.relu(x)
        x = model.maxpool(x)
        x = model.layer1(x)
        x = model.layer2(x)
        x = model.layer3(x)
        x = model.layer4(x)
        x = model.avgpool(x)  # Global average pooling
        x = torch.flatten(x, 1)
    return x.cpu().numpy()


# Generate a dataset of diverse synthetic images
n_samples = 200
all_features = []
all_labels = []
all_images_small = []

np.random.seed(42)
for i in range(n_samples):
    img = np.zeros((224, 224, 3), dtype=np.float32)
    label = i % 4  # 4 categories

    if label == 0:  # Horizontal lines
        spacing = np.random.randint(10, 30)
        color = np.random.rand(3)
        for row in range(0, 224, spacing):
            img[row:row+spacing//2, :] = color
    elif label == 1:  # Vertical lines
        spacing = np.random.randint(10, 30)
        color = np.random.rand(3)
        for col in range(0, 224, spacing):
            img[:, col:col+spacing//2] = color
    elif label == 2:  # Circle
        cx = np.random.randint(60, 164)
        cy = np.random.randint(60, 164)
        radius = np.random.randint(30, 70)
        color = np.random.rand(3)
        for r in range(224):
            for c in range(224):
                if (r - cx)**2 + (c - cy)**2 < radius**2:
                    img[r, c] = color
    else:  # Random noise
        img = np.random.rand(224, 224, 3).astype(np.float32)

    x = torch.tensor(img.transpose(2, 0, 1)).unsqueeze(0).to(device)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
    x_norm = (x - mean) / std

    feat = extract_feature_vector(resnet, x_norm)
    all_features.append(feat[0])
    all_labels.append(label)
    all_images_small.append(img[::4, ::4])  # downsample for display

all_features = np.array(all_features)
all_labels = np.array(all_labels)

print(f'Extracted {all_features.shape[0]} feature vectors of dimension {all_features.shape[1]}')
print(f'Categories: H-lines={sum(all_labels==0)}, V-lines={sum(all_labels==1)}, '
      f'Circles={sum(all_labels==2)}, Noise={sum(all_labels==3)}')

In [ ]:
# Visualize feature vectors in 2D using PCA and t-SNE
pca = PCA(n_components=2)
features_pca = pca.fit_transform(all_features)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_tsne = tsne.fit_transform(all_features)

category_names = ['H-Lines', 'V-Lines', 'Circles', 'Noise']
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, features_2d, method in [(axes[0], features_pca, 'PCA'),
                                  (axes[1], features_tsne, 't-SNE')]:
    for label in range(4):
        mask = all_labels == label
        ax.scatter(features_2d[mask, 0], features_2d[mask, 1],
                   c=colors[label], label=category_names[label],
                   s=40, alpha=0.7, edgecolors='k', linewidths=0.5)

    ax.set_title(f'{method} of CNN Feature Vectors', fontsize=13)
    ax.set_xlabel(f'{method} Component 1')
    ax.set_ylabel(f'{method} Component 2')
    ax.legend(fontsize=10)

plt.suptitle('CNN Feature Vectors Naturally Cluster by Visual Category!\n'
             'These features can serve as RL state representations',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

explained_var = pca.explained_variance_ratio_
print(f'PCA explained variance: {explained_var[0]:.1%} + {explained_var[1]:.1%} = {sum(explained_var):.1%}')

### 4.3 Feature Distance as State Similarity

In RL, we want similar states to have similar features. The **cosine similarity** between feature vectors measures this:

$$\text{sim}(\phi_a, \phi_b) = \frac{\phi_a \cdot \phi_b}{\|\phi_a\| \|\phi_b\|}$$

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute pairwise similarities between category centroids
centroids = []
for label in range(4):
    mask = all_labels == label
    centroids.append(all_features[mask].mean(axis=0))
centroids = np.array(centroids)

sim_matrix = cosine_similarity(centroids)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Similarity matrix
im = axes[0].imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
axes[0].set_xticks(range(4))
axes[0].set_yticks(range(4))
axes[0].set_xticklabels(category_names, rotation=30, ha='right')
axes[0].set_yticklabels(category_names)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{sim_matrix[i,j]:.3f}', ha='center', va='center',
                     fontsize=11, fontweight='bold')
axes[0].set_title('Feature Cosine Similarity Between Categories')
plt.colorbar(im, ax=axes[0])

# Within-class vs between-class distances
within_dists = []
between_dists = []
for label in range(4):
    mask = all_labels == label
    feats = all_features[mask]
    within_sim = cosine_similarity(feats)
    within_dists.extend(within_sim[np.triu_indices(len(feats), k=1)])

    for other_label in range(label + 1, 4):
        other_mask = all_labels == other_label
        between_sim = cosine_similarity(feats, all_features[other_mask])
        between_dists.extend(between_sim.ravel())

axes[1].hist(within_dists, bins=50, alpha=0.6, label='Within-class', color='green', density=True)
axes[1].hist(between_dists, bins=50, alpha=0.6, label='Between-class', color='red', density=True)
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_ylabel('Density')
axes[1].set_title('Feature Separability')
axes[1].legend(fontsize=11)

plt.suptitle('CNN Features Create Well-Separated State Representations',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Mean within-class similarity:  {np.mean(within_dists):.3f}')
print(f'Mean between-class similarity: {np.mean(between_dists):.3f}')
print(f'Separation ratio: {np.mean(within_dists) / (np.mean(between_dists) + 1e-8):.2f}x')

---

## Part 5: Choosing the Right Layer for Feature Extraction

### 5.1 Trade-offs

| Layer | Dimension | Semantics | Use Case |
|-------|-----------|-----------|----------|
| Early (layer1) | High-dim, spatial | Edges, textures | Fine-grained RL (manipulation) |
| Middle (layer2-3) | Medium | Parts, patterns | General purpose |
| Late (layer4 + GAP) | 512-d vector | Object-level | High-level RL (navigation) |

### 5.2 Practical Guidelines for RL

1. **Atari/simple games**: Often train CNN end-to-end with RL (no pre-trained features)
2. **Robot manipulation**: Pre-trained features + RL on top (faster convergence)
3. **Navigation**: Pre-trained ResNet features work well as state
4. **Custom domains**: Fine-tune the CNN with RL rewards

In [ ]:
# Compare feature quality at different layers
layers_compare = {
    'layer1': None,
    'layer2': None,
    'layer3': None,
    'layer4': None,
}

for layer_name in layers_compare:
    ext = FeatureExtractor(resnet, [layer_name])
    features_layer = []

    for img in test_images:
        x = torch.tensor(img.transpose(2, 0, 1)).unsqueeze(0).to(device)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
        x_norm = (x - mean) / std

        _, features = ext(x_norm)
        feat = features[layer_name].cpu().numpy()[0]
        # Global average pool to get a vector
        feat_vec = feat.mean(axis=(1, 2))
        features_layer.append(feat_vec)

    ext.remove_hooks()
    layers_compare[layer_name] = np.array(features_layer)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (layer_name, feats) in zip(axes, layers_compare.items()):
    sim = cosine_similarity(feats)
    im = ax.imshow(sim, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(test_titles)))
    ax.set_yticks(range(len(test_titles)))
    ax.set_xticklabels([t[:8] for t in test_titles], rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels([t[:8] for t in test_titles], fontsize=8)
    ax.set_title(f'{layer_name} (dim={feats.shape[1]})', fontsize=11)
    for i in range(len(test_titles)):
        for j in range(len(test_titles)):
            ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=8)

plt.suptitle('Feature Similarity at Different Layers\n'
             '(deeper layers show more semantic separation)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 🔑 Key Takeaways

| Concept | Key Insight |
|---------|------------|
| **CNN as feature extractor** | Remove classifier head → extract intermediate representations |
| **Layer hierarchy** | Early layers = edges/textures, deep layers = objects/semantics |
| **Grad-CAM** | $L^c = \text{ReLU}\left(\sum_k \alpha_k^c A^k\right)$ — gradient-weighted attention maps |
| **Feature vectors** | Compact state representations for RL: $\phi(x) \in \mathbb{R}^{512}$ |
| **For RL** | CNN features cluster by visual similarity → effective RL state |

### Connection to RL

In Deep RL:
- **DQN/A3C**: Train CNN jointly with RL objective from raw pixels
- **Sim-to-Real**: Use pre-trained features for domain transfer
- **CURL/DrQ**: Contrastive learning on CNN features for sample-efficient RL
- **Grad-CAM in RL**: Interpret what the policy network attends to during decision-making

---

## ➡️ Next: Module 5.4 — Transfer Learning

Instead of training CNNs from scratch, we can leverage **pre-trained models** (ResNet, VGG) and adapt them for our tasks — dramatically reducing training time and data requirements.